# Unsloth - Lab 3

## Notes
- Google Colab: Change runtime type - T4 GPU
- Use an Unsloth Model: [Huggingface Unsloth Collection](https://huggingface.co/unsloth/collections)
- Import data for `Fine-Tuning`
  - (Optional) Use `ChatGPT` or `Claude` to create data
  - (Optional) Use `Kaggle` for data
- `Modelfile` is for `Ollama` deployment

In [1]:
# Install Dependencies:
!pip install -q unsloth
!pip install -q --no-deps transformers accelerate bitsandbytes trl peft datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8

In [2]:
import torch
from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import os

file_path = "people_data.json"

if os.path.exists(file_path):
    print("File already exists — skipping download")
else:
    print("Downloading file...")
    !wget https://raw.githubusercontent.com/NeuralNine/youtube-tutorials/main/Unsloth%20Fine-Tuning/people_data.json

--2026-04-09 21:44:56--  https://raw.githubusercontent.com/NeuralNine/youtube-tutorials/main/Unsloth%20Fine-Tuning/people_data.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 87118 (85K) [text/plain]
Saving to: ‘people_data.json’

people_data.json    100%[===================>]  85.08K  --.-KB/s    in 0.002s  

2026-04-09 21:44:57 (45.0 MB/s) - ‘people_data.json’ saved [87118/87118]



In [ ]:
## 1. Config

In [4]:
# ------------------------------
# 1. Config
# ------------------------------
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
DTYPE = None  # let Unsloth auto-detect
SEED = 42

OUTPUT_DIR = "/content/unsloth_outputs"
DATA_FILE = "/content/people_data.json"   # update if needed

BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 2
EVAL_STEPS = 20
SAVE_STEPS = 20
LOGGING_STEPS = 10

In [ ]:
## 2. Load model + tokenizer

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
## 3. Load dataset

In [6]:
dataset = load_dataset(
    "json",
    data_files="/content/people_data.json",
    split="train",
)

required_cols = {"prompt", "response"}
missing = required_cols - set(dataset.column_names)
if missing:
    raise ValueError(f"Dataset missing required columns: {missing}")

Generating train split: 0 examples [00:00, ? examples/s]

## 4. Format dataset

In [7]:
def format_example(example):
    assistant_text = json.dumps(example["response"], ensure_ascii=False)

    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": assistant_text},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [8]:
# ------------------------------
# 5. Train / validation split
# ------------------------------
split_dataset = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['text'],
    num_rows: 270
})
Dataset({
    features: ['text'],
    num_rows: 30
})


In [9]:
# ------------------------------
# 6. Trainer
# ------------------------------
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        warmup_steps=5,
        logging_steps=LOGGING_STEPS,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        packing=False,  # set True if many short examples
    ),
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## 7. Train


In [10]:
train_result = trainer.train()
print(train_result)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 270 | Num Epochs = 2 | Total steps = 68
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
20,2.209814,1.656183
40,1.258429,1.131117
60,0.992347,0.952193
68,0.992347,0.934612


TrainOutput(global_step=68, training_loss=1.697235675419078, metrics={'train_runtime': 176.3575, 'train_samples_per_second': 3.062, 'train_steps_per_second': 0.386, 'total_flos': 869396757393408.0, 'train_loss': 1.697235675419078, 'epoch': 2.0})


## 8. Save adapter + tokenizer

In [11]:
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

print("Saved adapter + tokenizer.")

Saved adapter + tokenizer.


## 9. Inference helper

In [12]:
FastLanguageModel.for_inference(model)

def generate_response(prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    attention_mask = torch.ones_like(inputs)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Test Inference

In [13]:
test_prompt = "Extract the person's name and age from: Sarah is 29 years old."
response = generate_response(test_prompt)
print("\n=== MODEL OUTPUT ===")
print(response)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



=== MODEL OUTPUT ===
system

Cutting Knowledge Date: December 2023
Today Date: 09 Apr 2026

user

Extract the person's name and age from: Sarah is 29 years old.assistant

{"age": "29", "gender": "", "job": "", "name": "Sarah"}


## References:
- [Youtube: NueralNine - Fine-Tuning Local LLMs with Unsloth & Ollama](https://www.youtube.com/watch?v=W_xh6qNSfAQ)
- [Github: NeuralNine - Unsloth Fine-Tuning](https://github.com/NeuralNine/youtube-tutorials/tree/main/Unsloth%20Fine-Tuning)